# Lección 2.6 — Fuga de datos (*data leakage*): el error silencioso más caro

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 2.3 (preparación), 2.4 (ingeniería de características) y 2.5 (división de datos)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Definir qué es una **fuga de datos** y por qué produce modelos que fallan en la vida real.
- Reconocer los cuatro tipos principales: **fuga del objetivo**, **contaminación entre entrenamiento y prueba**, **grupos repartidos** y **fuga temporal**.
- Usar **pipelines** para que toda la preparación se aprenda solo con el entrenamiento.
- Aplicar una lista de verificación para cazar fugas antes de que lleguen a producción.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GroupKFold, KFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el pronóstico del paraguas

Un amigo presume que tiene un método infalible para predecir si llovió: **acierta el 98 % de las veces**. Su secreto: mira cuántos paraguas mojados hay en la entrada de la oficina.

El método "funciona"… pero es inútil para **pronosticar**, porque los paraguas mojados aparecen **después** de la lluvia. Tu amigo está usando una pista que contiene la respuesta.

Eso es una **fuga de datos** (*data leakage*): el modelo tiene acceso, durante el entrenamiento o la evaluación, a información que **no tendrá** cuando se use de verdad. El resultado:
- En el notebook, el modelo parece **excelente**.
- En la vida real, **fracasa**.

Es "silenciosa" porque nada da error: todo corre y los números son preciosos. Es "cara" porque se descubre tarde, cuando el modelo ya se usa para tomar decisiones.

### ✋ Ejercicio 1 — Calentamiento
¿En cuáles de estos casos hay una pista "tipo paraguas"?
1. Predecir si un estudiante aprobará el curso usando sus calificaciones de los **exámenes parciales**.
2. Predecir si un estudiante aprobará el curso usando si **recibió su diploma**.
3. Predecir si un cliente pagará su préstamo usando si su cuenta fue **enviada a cobranza**.
4. Predecir si un cliente pagará su préstamo usando su **historial de pagos anteriores**.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**2 y 3** son paraguas mojados: recibir el diploma y ser enviado a cobranza son **consecuencias** del resultado que quieres predecir. Se conocen *después*.

1 y 4 son válidas **si** se conocen antes del momento en que harás la predicción (los parciales antes del final; los pagos anteriores antes de otorgar el nuevo préstamo).
</details>

---
## 2. 📖 Tipo 1 — Fuga del objetivo: la columna que sabe demasiado

La forma más común: una columna que, de alguna manera, **contiene la respuesta**. Suele ser algo que se registra **después** del resultado, o **por causa** de él.

Una compañía de internet quiere predecir qué clientes **cancelarán**. Su base de datos tiene una columna `llamada_retencion`: si el área de retención llamó al cliente. Suena a una característica normal… pero esas llamadas se hacen **cuando el cliente ya pidió cancelar**.

In [ ]:
rng = np.random.default_rng(0)
n = 3_000
internet = pd.DataFrame({
    "meses_cliente":  rng.integers(1, 60, n),
    "cargo_mensual":  rng.integers(300, 1_200, n),
    "fallas_mes":     rng.poisson(1, n),
    "llamadas_soporte": rng.poisson(1.2, n),
})
riesgo = -1.5 - 0.04 * internet.meses_cliente + 0.002 * internet.cargo_mensual + 0.5 * internet.fallas_mes
internet["cancelo"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)
# El área de retención llama al 90 % de quienes piden cancelar, y a casi nadie más:
internet["llamada_retencion"] = np.where(internet.cancelo == 1, rng.random(n) < 0.9, rng.random(n) < 0.02).astype(int)

print(f"Cancelaron: {internet.cancelo.mean():.0%}")
internet.head()

### 🔮 Ejercicio 2 — Demasiado bueno para ser verdad
Entrenaremos un árbol pequeño con **todas** las columnas y otro **sin** `llamada_retencion`. Antes de ejecutar: ¿qué exactitud tendrá cada uno? ¿Cuál se parecerá más a lo que veremos en la vida real?

✍️ *Mi apuesta:*

In [ ]:
X_int = internet.drop(columns="cancelo")
y_int = internet["cancelo"]
X_tr, X_te, y_tr, y_te = train_test_split(X_int, y_int, test_size=0.25, stratify=y_int, random_state=0)

con_fuga = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
sin_fuga = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr.drop(columns="llamada_retencion"), y_tr)

print(f"Con llamada_retencion: {con_fuga.score(X_te, y_te):.1%}")
print(f"Sin llamada_retencion: {sin_fuga.score(X_te.drop(columns='llamada_retencion'), y_te):.1%}")
print(f"Referencia 'nadie cancela': {1 - y_te.mean():.1%}")
print("\nImportancia de cada columna (modelo con fuga):")
print(pd.Series(con_fuga.feature_importances_, index=X_int.columns).sort_values(ascending=False).round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

- **Con** la columna: ≈**95 %**, y la importancia de `llamada_retencion` es casi **1.0**: el modelo se apoya casi solo en ella.
- **Sin** ella: ≈**70 %**, apenas por encima de decir "nadie cancela" (≈66 %). Predecir conducta humana es difícil, y este número **honesto** es el que verías en la vida real.

¿Qué pasaría en producción con el modelo "con fuga"? Para un cliente que **todavía no ha pedido cancelar**, `llamada_retencion` vale 0 **siempre**. El modelo diría "no cancela" a todos: su 95 % se convierte en un modelo inútil.

**Señales de alerta que viste aquí:** un resultado espectacular en un problema difícil, y **una sola columna** que domina la importancia.
</details>

### ✋ Ejercicio 3 — La pregunta mágica
La pregunta que desenmascara casi cualquier fuga del objetivo es:

> **"¿Tendría yo este dato, con este valor, en el momento exacto en que necesito hacer la predicción?"**

Aplícala a estas columnas de un modelo que predice, **al recibir una solicitud**, si un préstamo terminará en impago:
1. `ingreso_declarado` · 2. `numero_de_recordatorios_de_pago_enviados` · 3. `tasa_de_interes_asignada` · 4. `edad` · 5. `saldo_pendiente_a_12_meses`

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. ✅ Se conoce al solicitar.
2. ❌ Los recordatorios se envían **después**, cuando el cliente se atrasa: es un paraguas mojado.
3. ⚠️ **Depende**: si la tasa se asigna en función del riesgo **ya evaluado** por otro modelo o por un analista, contiene parte de la respuesta… y además en el momento de la solicitud aún no existe. Hay que preguntar cómo y cuándo se genera.
4. ✅ Se conoce al solicitar.
5. ❌ Es información **del futuro** (12 meses después).

La 3 es el caso típico de la vida real: no es obviamente tramposa, y solo conociendo el **proceso del negocio** puedes decidir.
</details>

---
## 3. 📖 Tipo 2 — Contaminación: preparar con todos los datos

Esta fuga es más sutil. No hay ninguna columna tramposa; el error está en el **orden de los pasos**.

Cada paso de preparación **aprende algo de los datos**:
- El imputador aprende la **mediana** (Lección 2.2).
- El escalador aprende la **media** y la **desviación** (Lección 2.3).
- El *target encoder* aprende el **promedio de la etiqueta** por categoría (Lección 2.3).
- Un selector de características aprende **cuáles columnas se relacionan más con la etiqueta**.

Si cualquiera de esos pasos se aprende con **todos** los datos antes de dividir, el conjunto de prueba "le susurra" información al modelo. El examen deja de ser sorpresa.

### 🔮 Ejercicio 4 — Encontrar patrones en el ruido
Vamos a crear 100 pacientes con **5,000 mediciones al azar** y una etiqueta **también al azar**. No hay nada que aprender: lo honesto es ≈50 %.

Un analista hace esto:
1. Con **todos** los datos, elige las 20 columnas más relacionadas con la etiqueta.
2. **Después**, evalúa un modelo con validación cruzada usando solo esas 20.

Antes de ejecutar: ¿qué exactitud obtendrá?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(1)
X_ruido = rng.normal(size=(100, 5_000))     # 5,000 columnas sin ningún significado
y_ruido = rng.integers(0, 2, 100)            # etiquetas al azar

# ❌ MAL: seleccionar columnas mirando TODOS los datos, y después validar
selector = SelectKBest(f_classif, k=20).fit(X_ruido, y_ruido)
X_elegidas = selector.transform(X_ruido)
mal = cross_val_score(LogisticRegression(), X_elegidas, y_ruido, cv=5).mean()
print(f"❌ Seleccionar ANTES de validar      : {mal:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**90 %** con datos que son **puro ruido**. Un desastre que parece un éxito.

¿Qué pasó? Entre 5,000 columnas al azar, algunas se parecen a la etiqueta **por pura casualidad**. Al elegirlas mirando **todas** las filas (incluidas las que luego servirán de validación), el selector eligió justo las columnas que "aciertan" en esas filas. La validación ya estaba contaminada antes de empezar.
</details>

### 📖 La solución: el *pipeline*
Un **pipeline** (tubería) une la preparación y el modelo en **una sola pieza**. Cuando la validación cruzada (o tú) entrena el pipeline, **cada paso aprende solo con los datos de entrenamiento** de ese turno; los datos de validación solo se **transforman** con lo aprendido.

```
❌ MAL:   preparar(TODOS)  →  dividir  →  entrenar  →  evaluar
✅ BIEN:  dividir  →  preparar(entrenamiento)  →  entrenar  →  aplicar la MISMA preparación a prueba  →  evaluar
           └──────────────── el pipeline hace esto por ti ────────────────┘
```

In [ ]:
# ✅ BIEN: la selección va DENTRO del pipeline, así se repite en cada turno solo con el entrenamiento
tuberia = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
bien = cross_val_score(tuberia, X_ruido, y_ruido, cv=5).mean()
print(f"✅ Seleccionar DENTRO del pipeline   : {bien:.0%}")

### ✋ Ejercicio 5
1. ¿Qué exactitud dio el pipeline? ¿Es la "correcta" para datos al azar?
2. ¿Cambió el modelo? ¿Cambiaron los datos? Entonces, ¿qué cambió?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. ≈**47 %**, es decir, ≈50 % (puede salir algo por encima o por debajo por azar): lo que corresponde a datos sin patrón. Ahora la medición es **honesta**.
2. Ni el modelo ni los datos cambiaron. Cambió **el orden**: ahora la selección de columnas se hace **sin ver** las filas de validación.

> 💡 Con el escalado o la imputación el efecto suele ser **pequeño** (las medias y medianas cambian poco), pero con la **selección de columnas** y el **target encoding** puede ser enorme. La buena noticia: con un pipeline, **todos** quedan protegidos de la misma forma. Úsalo siempre.
</details>

### 📖 Así se ve un pipeline completo
Este es el patrón que usarás desde ahora. Fíjate en que `fit` solo recibe el entrenamiento.

In [ ]:
# Datos de ejemplo con huecos
rng = np.random.default_rng(2)
X_demo = pd.DataFrame(rng.normal(50, 10, size=(500, 4)), columns=["a", "b", "c", "d"])
y_demo = (X_demo.a + X_demo.b + rng.normal(0, 10, 500) > 100).astype(int)
X_demo[rng.random((500, 4)) < 0.1] = np.nan

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_demo, y_demo, test_size=0.25, random_state=0)

modelo = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
modelo.fit(X_tr_d, y_tr_d)                       # ← la mediana y la media se aprenden SOLO aquí
print(f"Exactitud en prueba: {modelo.score(X_te_d, y_te_d):.1%}")
print("Medianas que aprendió el imputador (solo con entrenamiento):", modelo[0].statistics_.round(1))

### 🔮 Ejercicio 6 — ¿Dónde está el error?
Un compañero escribió este código. Sin ejecutarlo, ¿hay fuga? ¿Dónde?
```python
escalador = StandardScaler().fit(X)                 # 1
X_escalado = escalador.transform(X)                 # 2
X_tr, X_te, y_tr, y_te = train_test_split(X_escalado, y)   # 3
modelo = KNeighborsClassifier().fit(X_tr, y_tr)     # 4
print(modelo.score(X_te, y_te))                     # 5
```

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí, en la línea 1**: el escalador aprende la media y la desviación con **todos** los datos, incluidos los que en la línea 3 terminarán en prueba. Arreglo: dividir **primero** y usar `make_pipeline(StandardScaler(), KNeighborsClassifier()).fit(X_tr, y_tr)`.

En este caso concreto la fuga probablemente sea pequeña, pero es un mal hábito: el día que el escalador sea un selector o un *target encoder*, el resultado será falso.
</details>

---
## 4. 📖 Tipo 3 — Grupos repartidos: reconocer en lugar de aprender

Si la misma persona (o tienda, o máquina) aparece en **varias filas**, y algunas caen en entrenamiento y otras en prueba, el modelo puede aprender a **reconocer a la persona** en vez de aprender el patrón general. Es como un examen que repite preguntas de la práctica con otras palabras.

Un hospital tiene **8 visitas por paciente** de 150 pacientes. Cada paciente tiene su "huella" (mediciones muy parecidas en todas sus visitas) y queremos predecir un diagnóstico que es **del paciente**.

### 🔮 Ejercicio 7 — ¿Aprende o reconoce?
Evaluaremos un KNN de dos maneras:
- **A:** validación cruzada **al azar** (las visitas de un paciente se reparten entre entrenamiento y validación).
- **B:** validación cruzada **por grupos** (`GroupKFold`): todas las visitas de un paciente van juntas.

Antes de ejecutar: ¿cuál dará más exactitud? ¿Cuál dice la verdad sobre un paciente **nuevo**?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(3)
pacientes, visitas = 150, 8
huella = rng.normal(size=(pacientes, 10))                         # cada paciente es distinto
diagnostico = (huella[:, 0] + rng.normal(0, 1.5, pacientes) > 0).astype(int)   # depende poco de las mediciones

X_vis = np.repeat(huella, visitas, axis=0) + rng.normal(0, 0.2, size=(pacientes * visitas, 10))
y_vis = np.repeat(diagnostico, visitas)
grupo = np.repeat(np.arange(pacientes), visitas)                  # qué paciente es cada fila

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
al_azar = cross_val_score(knn, X_vis, y_vis, cv=KFold(5, shuffle=True, random_state=0)).mean()
por_grupo = cross_val_score(knn, X_vis, y_vis, cv=GroupKFold(5), groups=grupo).mean()
print(f"A · Validación al azar  : {al_azar:.0%}")
print(f"B · Validación por grupo: {por_grupo:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **A (al azar): ≈100 %.** Para cada visita de validación, el vecino más cercano es… **otra visita del mismo paciente** que quedó en entrenamiento. El modelo no sabe nada de la enfermedad: **reconoce caras**.
- **B (por grupo): ≈58 %.** Para un paciente que nunca vio, el modelo solo puede usar lo que de verdad aprendió del diagnóstico, que es poco.

La B es la que importa: en el hospital, el modelo se usará con **pacientes nuevos**. Regla: **si hay grupos, divide por grupos** (`GroupKFold`, `GroupShuffleSplit`).

> 📝 Los **duplicados** (Lección 2.2) son el caso extremo de este problema: una copia exacta en entrenamiento y otra en prueba. Por eso se eliminan **antes** de dividir.
</details>

---
## 5. 📖 Tipo 4 — Fuga temporal: ver el futuro

Con datos en el tiempo, revolver al azar permite que el modelo entrene con datos **posteriores** a los que evalúa (Lección 2.5). Hay otras formas más escondidas:
- Una característica calculada con **toda** la historia: "promedio de ventas del cliente" que incluye las ventas de los meses que vas a predecir.
- Un dato que se **corrige después**: el inventario "final" del día, que se actualiza a medianoche, usado para predecir ventas de ese mismo día.
- Una tabla que se **sobrescribe**: si la base de datos guarda solo el valor **actual** de la dirección del cliente, no sabes dónde vivía cuando hizo cada compra.

### ✋ Ejercicio 8 — ¿Fuga temporal?
Una cadena de cafeterías predice las ventas de **mañana**. ¿Cuáles de estas características tienen fuga temporal?
1. Ventas de **ayer**.
2. Promedio de ventas del **mes completo** (incluido mañana).
3. Si mañana es feriado.
4. El clima **real** registrado mañana.
5. El **pronóstico** del clima para mañana, emitido hoy.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Con fuga: 2 y 4.** El promedio del mes completo incluye el día que quieres predecir. El clima real de mañana solo se conoce mañana.
- **Válidas: 1, 3 y 5.** Las ventas de ayer ya pasaron; el calendario de feriados se conoce de antemano; el pronóstico existe hoy (aunque se equivoque, es **lo que tendrás** al predecir).

La 4 vs. la 5 es una lección fina: entrenar con el clima **real** hace que el modelo dependa de una información perfecta que en producción **nunca** tendrá. Entrénalo con lo mismo que tendrá en producción: el pronóstico.
</details>

---
## 6. 📖 Lista de verificación anti-fugas

| ✅ Pregunta | Si la respuesta es "no" o "no sé"… |
|---|---|
| ¿Tendría cada columna, con ese valor, **en el momento** de predecir? | Quitarla o recalcularla con datos de ese momento |
| ¿Dividí los datos **antes** de cualquier paso que aprenda algo (imputar, escalar, codificar, seleccionar)? | Meter toda la preparación en un **pipeline** |
| ¿Eliminé los **duplicados** antes de dividir? | Hacerlo (Lección 2.2) |
| ¿Hay **grupos** (pacientes, clientes, tiendas)? ¿Van juntos? | Dividir por grupo |
| ¿Hay **tiempo**? ¿Entreno con el pasado y evalúo con el futuro? | Dividir por fecha |
| ¿Hay identificadores (ID de cliente, número de fila)? | Quitarlos: no son características (Lección 5.4 te mostrará un caso real) |
| ¿El resultado es **sospechosamente bueno** o **una sola columna domina**? | Investigar antes de celebrar |

### 🔮 Ejercicio 9 — El caso de las radiografías
Un modelo para detectar neumonía en radiografías alcanza un 97 % en prueba. Al investigar, descubren que las radiografías de pacientes graves se tomaron con un **aparato portátil** (en la cama del hospital), que deja una marca característica en la esquina de la imagen. ¿Es una fuga? ¿De qué tipo?

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí**: es una fuga del objetivo "disfrazada". El modelo aprendió a detectar **la marca del aparato portátil**, que se asocia con pacientes graves por **cómo se recogieron los datos**, no por la enfermedad. En un hospital donde todos usen el mismo aparato, el modelo fallará.

Este caso está basado en estudios reales con modelos de imágenes médicas. La lección: las fugas no siempre son columnas; pueden estar **escondidas en cómo se generaron los datos**.
</details>

---
## 7. 🧠 Resumen en 5 puntos
1. **Fuga de datos** = el modelo usa información que no tendrá en la vida real. Resultado: brillante en el notebook, inútil en producción.
2. **Fuga del objetivo:** columnas que son consecuencia del resultado ("paraguas mojados"). Pregunta mágica: *¿tendría este dato en el momento de predecir?*
3. **Contaminación:** cualquier paso que aprende de los datos (imputar, escalar, codificar, seleccionar) se aprende **solo con el entrenamiento**. La herramienta: el **pipeline**.
4. **Grupos y tiempo:** divide por paciente/cliente y por fecha cuando corresponda; quita duplicados antes de dividir.
5. **Desconfía de lo demasiado bueno:** un resultado espectacular o una sola columna dominante son señales de alerta.

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál de estos flujos tiene fuga?
- a) Dividir → escalar con entrenamiento → entrenar · b) Escalar con todos → dividir → entrenar · c) Pipeline(escalar, modelo) con validación cruzada · d) Ninguno

**2.** Para predecir si un paciente tiene diabetes, alguien usa la columna `toma_insulina`. ¿Qué tipo de problema es?
- a) Atípico · b) Fuga del objetivo · c) Desbalance · d) Ninguno

**3.** Tienes 50 fotos de cada uno de 40 perros y quieres un modelo que reconozca la raza de perros **nuevos**. ¿Cómo divides?
- a) Al azar · b) Por perro (todas las fotos de un perro al mismo lado) · c) Por fecha · d) Da igual

**4.** Con 5,000 columnas al azar y 100 filas, seleccionar las 20 "mejores" **antes** de validar da ≈85 %. ¿Cuál es la exactitud real?
- a) 85 % · b) ≈50 % · c) 100 % · d) No se puede saber

**5.** ¿Cuál es la mejor señal de que puede haber una fuga?
- a) El modelo tarda mucho en entrenar · b) La exactitud es sospechosamente alta y una sola columna domina · c) Hay datos faltantes · d) Las columnas están en escalas distintas

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El escalador aprendió con los datos que luego serán de prueba.
2. **b)** Tomar insulina es, casi siempre, **consecuencia** del diagnóstico de diabetes.
3. **b)** Si fotos del mismo perro quedan en ambos lados, el modelo reconoce **al perro**, no la raza.
4. **b)** Los datos son al azar: no hay nada que aprender.
5. **b)** Es la combinación clásica de una fuga del objetivo.
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué decimos que la fuga de datos es "más cara" que, por ejemplo, un modelo con sobreajuste evidente?

**B. Caso de análisis 💼** Un banco entrena un modelo para detectar transacciones fraudulentas. La columna `estado_cuenta` toma los valores "activa", "bloqueada" y "cerrada", según el estado **actual** de la cuenta en la base de datos. El modelo da 99 %. ¿Qué sospechas? ¿Cómo lo comprobarías y qué harías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque el sobreajuste **se ve**: el entrenamiento da 99 % y la prueba 70 %, y lo corriges antes de usar el modelo. La fuga **no se ve**: la prueba también da un número espectacular, así que el modelo se aprueba, se usa para decidir y el error aparece meses después, con dinero o personas afectadas.

**B.** Sospecha fuerte: cuando se detecta un fraude, la cuenta **se bloquea o se cierra**. El estado "actual" es una consecuencia del fraude (paraguas mojado) y además es **información del futuro** respecto a la transacción. Para comprobarlo: mirar la importancia de las columnas, entrenar sin `estado_cuenta` y ver si la exactitud se desploma, y preguntar al equipo de datos **cuándo** se actualiza esa columna. Solución: quitarla, o reconstruir el estado de la cuenta **en el momento** de cada transacción.
</details>

### 8.3 🧩 Práctica integradora: "El modelo milagroso del hospital"

Un hospital quiere predecir, **al momento del alta**, qué pacientes **volverán a internarse** en los próximos 30 días (para darles seguimiento). Un analista anterior dejó un modelo "milagroso" y te piden revisarlo antes de ponerlo en uso.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos y el modelo del analista**

In [ ]:
rng = np.random.default_rng(10)
n_pac = 800
pac = pd.DataFrame({
    "id_paciente":        np.arange(n_pac),
    "edad":               rng.integers(20, 90, n_pac),
    "internaciones_previas": rng.poisson(1, n_pac),
    "enfermedades_cronicas": rng.integers(0, 5, n_pac),
})
riesgo = -3.5 + 0.03 * pac.edad + 0.5 * pac.internaciones_previas + 0.3 * pac.enfermedades_cronicas
pac["reingreso"] = (rng.random(n_pac) < 1 / (1 + np.exp(-riesgo))).astype(int)

# Cada paciente tuvo de 1 a 4 altas; las de un mismo paciente se parecen mucho
filas = pac.loc[pac.index.repeat(rng.integers(1, 5, n_pac))].reset_index(drop=True)
m = len(filas)
filas["dias_internado"] = (3 + 2 * filas.enfermedades_cronicas + rng.poisson(2, m)).astype(float)
filas["medicamentos_al_alta"] = (2 + filas.enfermedades_cronicas + rng.poisson(1, m)).astype(float)
filas["costo_anual_total"] = (50_000 * filas.dias_internado + 400_000 * filas.reingreso
                              + rng.normal(0, 50_000, m)).round(-3)     # ⚠️ se calcula al cierre del año
filas.loc[rng.random(m) < 0.08, "medicamentos_al_alta"] = np.nan

X_h = filas.drop(columns="reingreso")
y_h = filas["reingreso"]
print(f"{m} altas de {n_pac} pacientes · reingresaron: {y_h.mean():.0%}")

# El modelo del analista: rellena huecos con TODOS los datos, divide al azar y entrena
X_relleno = X_h.fillna(X_h.median())
Xa, Xb, ya, yb = train_test_split(X_relleno, y_h, test_size=0.25, random_state=0)
arbol_analista = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xa, ya)
print(f"Exactitud del modelo del analista: {arbol_analista.score(Xb, yb):.1%}")
filas.head()

**Paso 2 ✋ — Busca al culpable.** Mira la importancia de las columnas del modelo del analista. ¿Cuál domina? Aplica la pregunta mágica: ¿se conoce **al momento del alta**?

In [ ]:
pd.Series(arbol_analista.feature_importances_, index=X_h.columns).sort_values(ascending=False).round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Domina **`costo_anual_total`** (importancia ≈0.78). Se calcula **al cierre del año** e incluye el costo de la reinternación: un paciente que reingresó cuesta mucho más. Es un paraguas mojado. Al momento del alta, ese dato **no existe**.

¿Y `dias_internado` en segundo lugar (≈0.22)? El árbol lo usa como **cómplice**: el costo depende de los días internado *y* del reingreso, así que, sabiendo los días, puede "descontar" esa parte y quedarse con la del reingreso. `dias_internado` es una columna legítima; el problema es su socio.

¿Notaste también `id_paciente`? Un identificador **nunca** debe ser característica; aquí tiene poca importancia, pero hay que quitarlo igual.
</details>

**Paso 3 ✏️ — Quita las columnas tramposas.** Completa la lista y vuelve a entrenar (todavía con la división al azar del analista).

In [ ]:
COLUMNAS_A_QUITAR = ["costo_anual_total", "id_paciente"]     # ✏️

Xa2, Xb2 = Xa.drop(columns=COLUMNAS_A_QUITAR), Xb.drop(columns=COLUMNAS_A_QUITAR)
arbol2 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xa2, ya)
print(f"Sin columnas tramposas: {arbol2.score(Xb2, yb):.1%}")
print(f"Referencia 'nadie reingresa': {1 - yb.mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

La exactitud cae de ≈**99 %** a ≈**72 %**, apenas por encima de "nadie reingresa" (≈68 %). Duele, pero ahora el número se parece más a la realidad. Y todavía no terminamos.
</details>

**Paso 4 🔮 — ¿Y los pacientes repetidos?** Un mismo paciente aparece en varias filas (sus distintas altas), y la división al azar los reparte. Además, el analista rellenó los huecos con la mediana de **todos** los datos. Arreglemos las dos cosas: **pipeline** + **validación por grupos** (por `id_paciente`).

Antes de ejecutar: ¿cuál dará más exactitud, la validación al azar o la validación por grupo? ¿Cuál dice la verdad sobre pacientes nuevos?

✍️ *Mi apuesta:*

In [ ]:
X_honesto = X_h.drop(columns=COLUMNAS_A_QUITAR)
grupos = X_h["id_paciente"]

tuberia_h = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=5, random_state=0))
azar = cross_val_score(tuberia_h, X_honesto, y_h, cv=KFold(5, shuffle=True, random_state=0)).mean()
grupo = cross_val_score(tuberia_h, X_honesto, y_h, cv=GroupKFold(5), groups=grupos).mean()
print(f"Pipeline + validación al azar   : {azar:.1%}")
print(f"Pipeline + validación por grupo : {grupo:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**Al azar ≈75 %, por grupo ≈71 %.** Con la división al azar, el árbol "reconoce" a pacientes que ya vio en otra de sus altas (misma edad, mismas enfermedades, días internado parecidos). Por grupo, cada paciente de validación es **nuevo**, como en la vida real: ese ≈71 % es el número honesto. (No lo compares directamente con el 72 % del paso 3: aquel salía de **un solo** reparto; estos promedian 5.)

(El efecto de la imputación con todos los datos es aquí pequeño, pero el pipeline lo elimina de raíz.)
</details>

**Paso 5 ✋ — Tu informe al hospital.** Resume en **tres frases**, sin tecnicismos: qué estaba mal en el modelo milagroso, cuál es la expectativa realista y qué recomiendas.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

*"El modelo anterior parecía acertar el 99 % porque usaba el costo anual del paciente, un dato que solo se conoce a fin de año y que ya incluye el costo del reingreso: en la práctica no podría usarse. Corregido ese problema, y evaluándolo con pacientes que el modelo nunca vio, acierta alrededor del 71 %, apenas por encima de suponer que nadie reingresa (68 %). Recomiendo no usarlo todavía para decisiones y, en cambio, buscar más información disponible al momento del alta (resultados de laboratorio, apoyo familiar, citas de seguimiento) y medir su desempeño con las métricas adecuadas (Lección 4.2)."*

Fíjate: **un resultado honesto y modesto vale mucho más que uno espectacular y falso.**
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, *Common pitfalls and recommended practices* (sección *Data leakage*).
- 📘 **Artículo:** Kaufman et al., "Leakage in Data Mining: Formulation, Detection, and Avoidance" (2012), el clásico sobre el tema (inglés).
- 📘 **Kaggle Learn:** curso *Intermediate Machine Learning*, lección *Data Leakage* (inglés).
- 📰 **Caso real:** busca "Zech et al. 2018 pneumonia radiographs confounding" (el caso del Ejercicio 9).

---
⬅️ **Anterior:** [2.5 División de datos: entrenamiento, validación y prueba](2.5-division-de-datos.ipynb) · ➡️ **Siguiente:** [2.7 Datos desbalanceados: cuando lo importante es raro](2.7-datos-desbalanceados.ipynb)